# Notebook 11: Kalshi Market Comparison (2025 Season)

Secondary evaluation track: benchmarks model predictions against Kalshi implied probabilities for 2025 season games.

**Two-track reporting:** This is a partial benchmark (2025 games with Kalshi data only), distinct from the primary 2015-2024 backtest in notebooks 09/10. Results are never conflated.

**Fold:** Train 2015-2023, calibrate 2024, predict 2025 (core feature set only).

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from sklearn.metrics import brier_score_loss

from src.features.feature_builder import FeatureBuilder
from src.models.predict import predict_2025
from src.data.kalshi import fetch_kalshi_markets, fetch_kalshi_open_prices
from src.models.evaluate import plot_calibration_curves

## Step 1: Build 2025 Feature Matrix

Extends the feature matrix through 2025 using FeatureBuilder. This will ingest 2025 team batting, SP stats, game logs, and schedule data from pybaseball/MLB Stats API (cached after first run).

In [ ]:
fb = FeatureBuilder(seasons=list(range(2015, 2026)))
feature_matrix = fb.build()
print(f"Feature matrix: {len(feature_matrix)} rows, seasons {feature_matrix['season'].min()}-{feature_matrix['season'].max()}")
print(f"2025 games: {(feature_matrix['season'] == 2025).sum()}")

## Step 2: Generate 2025 Predictions

Runs 3 models (LR, RF, XGBoost) on the 2025 fold using core feature set only (xwoba_diff excluded per Phase 3 finding).

In [ ]:
predictions_2025 = predict_2025(feature_matrix)
predictions_2025.to_parquet("../data/results/predictions_2025.parquet", index=False)
print(f"Predictions: {len(predictions_2025)} rows ({len(predictions_2025) // 3} games x 3 models)")
print(f"Models: {sorted(predictions_2025['model_name'].unique())}")
print(f"Feature set: {predictions_2025['feature_set'].unique()}")
print(f"Fold test year: {predictions_2025['fold_test_year'].unique()}")

## Step 3: Fetch Kalshi Opening Prices

Fetches pre-game opening prices via Kalshi batch candlestick API (`price.open_dollars`). Markets with no trading volume get NaN and fall back to closing price with a caveat.

In [ ]:
kalshi_df = fetch_kalshi_markets(max_age_hours=float("inf"))  # Use cached data
kalshi_df = fetch_kalshi_open_prices(kalshi_df)

n_open = kalshi_df["kalshi_open_price"].notna().sum()
n_fallback = kalshi_df["kalshi_open_price"].isna().sum()
print(f"Kalshi markets: {len(kalshi_df)} total")
print(f"  Opening price available: {n_open} ({100*n_open/len(kalshi_df):.1f}%)")
print(f"  Fallback to closing price: {n_fallback} ({100*n_fallback/len(kalshi_df):.1f}%)")

# Fill NaN open prices with closing price (fallback)
kalshi_df["kalshi_open_price"] = kalshi_df["kalshi_open_price"].fillna(kalshi_df["kalshi_yes_price"])
# Track which rows are fallback
kalshi_df["is_fallback_price"] = kalshi_df["kalshi_open_price"] == kalshi_df["kalshi_yes_price"]

## Step 4: Join Predictions to Kalshi Data

Join on `(game_date, home_team, away_team)` with date type alignment. Doubleheader games (9 in 2025) are dropped from the comparison to avoid many-to-many join issues (<0.5% of data).

In [ ]:
# Date type alignment: Kalshi date is string, predictions game_date is datetime
kalshi_df["game_date"] = pd.to_datetime(kalshi_df["date"])

# Drop doubleheader duplicates from Kalshi data (keep first occurrence per date+teams)
kalshi_deduped = kalshi_df.drop_duplicates(subset=["game_date", "home_team", "away_team"], keep="first")
n_dh_dropped = len(kalshi_df) - len(kalshi_deduped)
if n_dh_dropped > 0:
    print(f"Dropped {n_dh_dropped} doubleheader duplicate(s) from Kalshi data")

# Drop doubleheader duplicates from predictions too
pred_deduped = predictions_2025.drop_duplicates(
    subset=["game_date", "home_team", "away_team", "model_name"], keep="first"
)

# Merge
merged = pred_deduped.merge(
    kalshi_deduped[["game_date", "home_team", "away_team", "kalshi_open_price", "kalshi_yes_price", "is_fallback_price"]],
    on=["game_date", "home_team", "away_team"],
    how="inner",
)
print(f"Matched: {len(merged)} rows ({len(merged) // 3} games x 3 models)")
print(f"  Fallback price rows: {merged['is_fallback_price'].sum()}")

## Step 5: Brier Score Comparison (Model vs Kalshi)

Computes Brier score for each model's calibrated probabilities and for Kalshi opening prices, on the same set of 2025 games. Lower is better.

This is a **partial benchmark** covering 2025 season only -- not comparable to the 2015-2024 aggregate backtest in notebook 10.

In [ ]:
# Compute Kalshi Brier score (opening price as probability)
# Use one row per game (not per model) for Kalshi baseline
games_df = merged[merged["model_name"] == "lr"]  # Any model works -- same game set
kalshi_brier = brier_score_loss(games_df["home_win"], games_df["kalshi_open_price"])

# Compute model Brier scores
brier_rows = []
for model_name in ["lr", "rf", "xgb"]:
    model_df = merged[merged["model_name"] == model_name]
    brier = brier_score_loss(model_df["home_win"], model_df["prob_calibrated"])
    brier_rows.append({
        "source": model_name,
        "brier_score": round(brier, 4),
        "n_games": len(model_df),
    })

brier_rows.append({
    "source": "kalshi_market",
    "brier_score": round(kalshi_brier, 4),
    "n_games": len(games_df),
})

brier_comparison = pd.DataFrame(brier_rows)
print("=== Brier Score Comparison: Model vs Kalshi (2025 Season) ===")
print(f"(Lower is better. Partial benchmark: {len(games_df)} games with Kalshi data.)")
print()
print(brier_comparison.to_string(index=False))

## Step 6: Calibration Curves (2025 Only)

Reliability diagrams for each model on 2025 games, showing how well-calibrated the probability outputs are for this single-season sample.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")

# Model calibration curves
for model_name in ["lr", "rf", "xgb"]:
    model_df = merged[merged["model_name"] == model_name]
    fraction_pos, mean_pred = calibration_curve(
        model_df["home_win"], model_df["prob_calibrated"], n_bins=10, strategy="uniform"
    )
    ax.plot(mean_pred, fraction_pos, "s-", label=f"{model_name} (model)")

# Kalshi calibration curve
fraction_pos_k, mean_pred_k = calibration_curve(
    games_df["home_win"], games_df["kalshi_open_price"], n_bins=10, strategy="uniform"
)
ax.plot(mean_pred_k, fraction_pos_k, "o--", label="Kalshi market", color="black")

ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curves: Models vs Kalshi (2025 Season)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

- **Evaluation track:** Secondary (2025 season only, partial Kalshi coverage)
- **Models:** LR, RF, XGBoost on core feature set
- **Benchmark:** Kalshi opening price (pre-game) as implied probability
- **Output:** `data/results/predictions_2025.parquet` (used by notebook 12 for edge analysis)